# NB09 - Export the deployment bundle + collect the final tables
Packages the 5 fold-models of one CV repeat (an ensemble), their temperature scalers, OOD statistics, the conformal quantile and rejection threshold
into `OASIS1_outputs/step9_bundle/` - the only thing the web app needs.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
MODEL = 'cnn3d'
b = PL.step9_export(cfg, paths, MODEL)
{k: b[k] for k in ['model_name','class_names','qhat','conf_tau','conformal_note']}

In [ ]:
import os, pandas as pd
for f in [f'{paths.evaluation}/{MODEL}/model_comparison.csv', f'{paths.evaluation}/{MODEL}/calibration_summary.csv', f'{paths.evaluation}/{MODEL}/age_matched_metrics.csv']:
    if os.path.exists(f): print(os.path.basename(f)); display(pd.read_csv(f))
print('All result files are under', paths.out_dir)